<a href="https://colab.research.google.com/github/HDN-Praharshini123/NLP/blob/main/experiment_22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install spaCy
!pip install -qq spacy

# Download a small English language model
# This model includes capabilities for tokenization, part-of-speech tagging, named entity recognition, and dependency parsing.
# The 'sm' (small) model is chosen for quicker download and basic functionality.
!python -m spacy download en_core_web_sm -q

import spacy

# Load the English language model
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("SpaCy model 'en_core_web_sm' not found. Please ensure it's downloaded by running the previous cell.")
    exit()

def perform_basic_reference_resolution(text):
    doc = nlp(text)

    # Store potential antecedents (named entities and nouns) and their positions
    antecedents = {}

    print(f"Analyzing text:\n'{text}'\n")
    print("--- Identified References ---")

    for token in doc:
        if token.pos_ == "PRON": # Check if the token is a pronoun
            # Simple heuristic: Look for the most recent named entity or noun as a potential antecedent
            potential_antecedent = None
            for ent in doc.ents:
                if ent.end <= token.i and (potential_antecedent is None or ent.end > potential_antecedent.end):
                    potential_antecedent = ent

            if potential_antecedent is None:
                # If no named entity, look for the most recent noun phrase
                for chunk in doc.noun_chunks:
                    if chunk.end <= token.i and (potential_antecedent is None or chunk.end > potential_antecedent.end):
                        potential_antecedent = chunk

            if potential_antecedent:
                print(f"Pronoun: '{token.text}' (Index: {token.i}) -> Potential Antecedent: '{potential_antecedent.text}' (Type: {potential_antecedent.label_ if hasattr(potential_antecedent, 'label_') else 'NOUN_CHUNK'}) (Index: {potential_antecedent.start}-{potential_antecedent.end-1})")
            else:
                print(f"Pronoun: '{token.text}' (Index: {token.i}) -> No clear antecedent found by simple heuristic")
        elif token.ent_type_ != "": # If it's a named entity, record it
            antecedents[token.i] = token.text
        elif token.pos_ == "NOUN": # If it's a noun, record it as a potential antecedent
            antecedents[token.i] = token.text

# Example Usage
text1 = "Alice met Bob. She told him about her new project. They were excited."
perform_basic_reference_resolution(text1)

print("\n" + "="*40 + "\n")

text2 = "The company announced its new product. It is expected to revolutionize the market."
perform_basic_reference_resolution(text2)

print("\n" + "="*40 + "\n")

text3 = "Dr. Smith is a brilliant scientist. He developed a new theory."
perform_basic_reference_resolution(text3)

print("\n--- Limitations ---")
print("This is a highly simplified rule-based approach and does not represent robust coreference resolution. It primarily links pronouns to the most recent preceding named entity or noun phrase. True coreference resolution requires more sophisticated models, often involving machine learning, to handle complex cases like: ")
print("1. Anaphoric resolution (e.g., 'the big dog' referring to 'a golden retriever').")
print("2. Bridging references (e.g., 'the car' and 'the engine').")
print("3. Different types of pronouns (e.g., possessive, demonstrative).")
print("4. Contextual understanding and world knowledge.")
print("For advanced coreference resolution, you would typically use libraries like `neuralcoref` (older `spaCy` versions) or explore models available through `spaCy-transformers` and Hugging Face for modern `spaCy` setups.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 95.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Analyzing text:
'Alice met Bob. She told him about her new project. They were excited.'

--- Identified References ---
Pronoun: 'She' (Index: 4) -> Potential Antecedent: 'Bob' (Type: PERSON) (Index: 2-2)
Pronoun: 'him' (Index: 6) -> Potential Antecedent: 'Bob' (Type: PERSON) (Index: 2-2)
Pronoun: 'her' (Index: 8) -> Potential Antecedent: 'Bob' (Type: PERSON) (Index: 2-2)
Pronoun: 'They' (Index: 12) -> Potential Antecedent: 'Bob' (Type: PERSON) (Index: 2-2)


Analyzing text:
'The company announced its new product. It is expected to revolutionize the market.'

--- Identified Ref